# MIRA: Multimodal Infrastructure Risk Analyzer
This project is a POC for ACM Research;
Lead: Advay Chandramouli


## Import Requisite Libraries

In [79]:
import pandas as pd
df = pd.read_csv("data/im3_open_source_data_center_atlas.csv")
df.shape
df.head()

,id,state,state_abb,state_id,county,county_id,operator,ref,name,sqft,lon,lat,type
0,2744301,New Jersey,NJ,34,Middlesex County,23,NaN,NaN,Verizon,105786.0,-74.496521,40.544256,building
1,7805491,Ohio,OH,39,Franklin County,49,NaN,NaN,Discover Financial Services New Albany,188209.0,-82.814358,40.100657,building
2,9474864,North Carolina,NC,37,Caldwell County,27,Google,NaN,Google Data Center,3407194.0,-81.546515,35.894738,campus
3,13924557,Iowa,IA,19,Polk County,153,Microsoft,NaN,Project Alluvion,10962475.0,-93.711719,41.515955,campus
4,14593270,North Carolina,NC,37,Catawba County,35,NaN,NaN,Apple - Maiden Data Center,5431080.0,-81.261809,35.588771,campus


## Exploratory Data Analysis (EDA)

In [80]:
unique_operators = sorted(
    df["operator"].fillna("").replace("", "(Not specified)").unique()
)
print(f"{len(unique_operators)} unique operators:")
pd.DataFrame(unique_operators, columns=["operator"])

130 unique operators:


,operator
0,(Not specified)
1,AT&T
2,Actapio
3,AiNET
4,Alabama Supercomputer Authority
...,...
125,Yahoo
126,Yosemite Community College District
127,bigbyte.cc
128,datasite


In [81]:
import plotly.express as px

# Records by Operator (top 20 for readability; long names work best on horizontal bars)
operator_counts = (
    df["operator"]
    .fillna("")
    .replace("", "(Not specified)")
    .value_counts()
    .reset_index()
)
operator_counts.columns = ["operator", "count"]
operator_counts["pct"] = (
    operator_counts["count"] / operator_counts["count"].sum() * 100
).round(1)
operator_top = operator_counts.head(20).sort_values("count")
operator_top["label"] = operator_top.apply(
    lambda row: f"{row['count']} ({row['pct']}%)", axis=1
)

fig_ops = px.bar(
    operator_top,
    x="count",
    y="operator",
    orientation="h",
    title="Records by Operator (Top 20)",
    labels={"count": "Number of Data Centers", "operator": "Operator"},
    text="label",
    color="count",
    color_continuous_scale="Blues",
)
fig_ops.update_layout(
    yaxis={"categoryorder": "total ascending"},
    showlegend=False,
    coloraxis_showscale=False,
    height=640,
    margin={"l": 20, "r": 40, "t": 60, "b": 40},
    plot_bgcolor="white",
)
fig_ops.update_traces(textposition="outside", cliponaxis=False)
fig_ops.show()

# Records by State (horizontal bar chart with counts and percentages)
state_counts = df["state"].value_counts().reset_index()
state_counts.columns = ["state", "count"]
state_counts["pct"] = (state_counts["count"] / state_counts["count"].sum() * 100).round(
    1
)
state_sorted = state_counts.sort_values("count")
state_sorted["label"] = state_sorted.apply(
    lambda row: f"{row['count']} ({row['pct']}%)", axis=1
)

fig_states = px.bar(
    state_sorted,
    x="count",
    y="state",
    orientation="h",
    title="Records by State",
    labels={"count": "Number of Data Centers", "state": "State"},
    text="label",
    color="count",
    color_continuous_scale="Teal",
)
fig_states.update_layout(
    yaxis={"categoryorder": "total ascending"},
    showlegend=False,
    coloraxis_showscale=False,
    height=1100,
    margin={"l": 20, "r": 40, "t": 60, "b": 40},
    plot_bgcolor="white",
)
fig_states.update_traces(textposition="outside", cliponaxis=False)
fig_states.show()

## Tabular Preprocessing

In [82]:
df = df[df["type"] == "building"]
df_buildings = df.drop(columns=["state_abb", "state_id", "county", "county_id", "ref"])
df_buildings["sqft"] = df_buildings["sqft"].astype(int)

In [83]:
MW_DENSITY_W_PER_SQFT = 150
MW_DIVISOR = 1_000_000

TIER_LABELS = {
    0: "Edge/Enterprise",
    1: "Colocation",
    2: "Hyperscale",
}

SQFT_COLO_MIN = 50_000
SQFT_HYPER_MIN = 200_000
MW_COLO_MIN = 5.0
MW_HYPER_MIN = 40.0


def assign_impact_tier(sqft: int, est_mw: float) -> int:
    if sqft >= SQFT_HYPER_MIN or est_mw >= MW_HYPER_MIN:
        return 2
    if sqft >= SQFT_COLO_MIN or est_mw >= MW_COLO_MIN:
        return 1
    return 0


df_buildings["est_mw"] = df_buildings["sqft"] * MW_DENSITY_W_PER_SQFT / MW_DIVISOR
df_buildings["impact_tier"] = df_buildings.apply(
    lambda r: assign_impact_tier(r["sqft"], r["est_mw"]), axis=1
)
df_buildings["impact_tier_label"] = df_buildings["impact_tier"].map(TIER_LABELS)

In [84]:
import plotly.express as px

tier_order = [TIER_LABELS[k] for k in sorted(TIER_LABELS)]

print("Impact tier distribution:")
print(df_buildings["impact_tier_label"].value_counts().reindex(tier_order).to_string())
print(f"\nTotal buildings: {len(df_buildings)}")

fig = px.histogram(
    df_buildings,
    x="est_mw",
    color="impact_tier_label",
    nbins=60,
    title="Estimated Power (MW) by Impact Tier",
    labels={"est_mw": "Estimated MW (150 W/sqft)", "impact_tier_label": "Tier"},
    category_orders={"impact_tier_label": tier_order},
    color_discrete_sequence=["#4C78A8", "#F58518", "#E45756"],
)
fig.update_layout(plot_bgcolor="white", height=400, margin={"t": 50, "b": 40})
fig.show()

Impact tier distribution:
impact_tier_label
Edge/Enterprise    181
Colocation         592
Hyperscale         267

Total buildings: 1040


In [85]:
print(f"Shape: {df_buildings.shape}")
print(f"Columns: {list(df_buildings.columns)}")
df_buildings.head()

Shape: (1040, 11)
Columns: ['id', 'state', 'operator', 'name', 'sqft', 'lon', 'lat', 'type', 'est_mw', 'impact_tier', 'impact_tier_label']


,id,state,operator,name,sqft,lon,lat,type,est_mw,impact_tier,impact_tier_label
0,2744301,New Jersey,NaN,Verizon,105786,-74.496521,40.544256,building,15.86790,1,Colocation
1,7805491,Ohio,NaN,Discover Financial Services New Albany,188209,-82.814358,40.100657,building,28.23135,1,Colocation
5,14930068,New Mexico,Sandia National Laboratories,HPC (880),158463,-106.542822,35.049942,building,23.76945,1,Colocation
7,15884451,New Jersey,Barclays,Barclays Datacenter,94373,-74.285481,40.643753,building,14.15595,1,Colocation
8,16282459,Maryland,AiNET,CyberNAP Glen Burnie,89879,-76.606431,39.140242,building,13.48185,1,Colocation


## WRI Aqueduct Input Format

In [86]:
example_coordinates = pd.read_csv("data/example_coordinates.csv")
print("Example coordinates format:")
example_coordinates

wri_aqueduct_df = df_buildings[["id", "name", "lat", "lon"]].copy()
wri_aqueduct_df.columns = example_coordinates.columns

print(f"Columns: {list(wri_aqueduct_df.columns)}")
print(f"Shape: {wri_aqueduct_df.shape}")
wri_aqueduct_df.head()

Example coordinates format:
Columns: ['id', 'location name', 'latitude (decimal degrees)', 'longitude (decimal degrees)']
Shape: (1040, 4)


,id,location name,latitude (decimal degrees),longitude (decimal degrees)
0,2744301,Verizon,40.544256,-74.496521
1,7805491,Discover Financial Services New Albany,40.100657,-82.814358
5,14930068,HPC (880),35.049942,-106.542822
7,15884451,Barclays Datacenter,40.643753,-74.285481
8,16282459,CyberNAP Glen Burnie,39.140242,-76.606431


In [87]:
from pathlib import Path

batch_dir = Path("data/wri_input_batches")
batch_dir.mkdir(parents=True, exist_ok=True)

batch_size = 500
batch_files = []

for batch_num, start in enumerate(range(0, len(wri_aqueduct_df), batch_size), start=1):
    batch_df = wri_aqueduct_df.iloc[start : start + batch_size].copy()
    batch_path = batch_dir / f"batch_{batch_num:03d}.csv"
    batch_df.to_csv(batch_path, index=False)
    batch_files.append((batch_path.name, len(batch_df)))

print(f"Created {len(batch_files)} batch(es) in {batch_dir}/")
for name, n_rows in batch_files:
    print(f"  {name}: {n_rows} rows")

Created 3 batch(es) in data/wri_input_batches/
  batch_001.csv: 500 rows
  batch_002.csv: 500 rows
  batch_003.csv: 40 rows


In [88]:
wri_top5_data = pd.read_csv("data/wri_top5_data.csv")

with open("data/columns.txt", "w") as f:
    f.write("\n".join(wri_top5_data.columns))

print(f"Columns ({len(wri_top5_data.columns)}):")
print(list(wri_top5_data.columns))

print("\nColumn types:")
wri_top5_data.dtypes

Columns (271):
['the_geom', 'points_id', 'location_name', 'input_address', 'match_address', 'latitude', 'longitude', 'major_basin_name', 'minor_basin_name', 'aquifer_name', 'string_id', 'aq30_id', 'pfaf_id', 'gid_1', 'aqid', 'gid_0', 'name_0', 'name_1', 'area_km2', 'bws_raw', 'bws_score', 'bws_cat', 'bws_label', 'bwd_raw', 'bwd_score', 'bwd_cat', 'bwd_label', 'iav_raw', 'iav_score', 'iav_cat', 'iav_label', 'sev_raw', 'sev_score', 'sev_cat', 'sev_label', 'gtd_raw', 'gtd_score', 'gtd_cat', 'gtd_label', 'rfr_raw', 'rfr_score', 'rfr_cat', 'rfr_label', 'cfr_raw', 'cfr_score', 'cfr_cat', 'cfr_label', 'drr_raw', 'drr_score', 'drr_cat', 'drr_label', 'ucw_raw', 'ucw_score', 'ucw_cat', 'ucw_label', 'cep_raw', 'cep_score', 'cep_cat', 'cep_label', 'udw_raw', 'udw_score', 'udw_cat', 'udw_label', 'usa_raw', 'usa_score', 'usa_cat', 'usa_label', 'rri_raw', 'rri_score', 'rri_cat', 'rri_label', 'w_awr_def_qan_raw', 'w_awr_def_qan_score', 'w_awr_def_qan_cat', 'w_awr_def_qan_label', 'w_awr_def_qan_weight_

the_geom                             str
points_id                          int64
location_name                        str
input_address                        str
match_address                        str
                                  ...   
w_awr_tex_tot_raw                float64
w_awr_tex_tot_score              float64
w_awr_tex_tot_cat                  int64
w_awr_tex_tot_label                  str
w_awr_tex_tot_weight_fraction    float64
Length: 271, dtype: object

## Combine WRI Source Batches

In [89]:
from pathlib import Path

source_dir = Path("data/wri_source_data")
batch_paths = sorted(source_dir.glob("batch_*.csv"))

print(f"Found {len(batch_paths)} batch file(s) in {source_dir}/")
for path in batch_paths:
    batch_df = pd.read_csv(path)
    print(f"  {path.name}: {len(batch_df)} rows, {len(batch_df.columns)} columns")

print(f"\nFirst batch columns ({len(pd.read_csv(batch_paths[0]).columns)}):")
print(list(pd.read_csv(batch_paths[0]).columns[:10]), "...")

Found 3 batch file(s) in data/wri_source_data/
  batch_001.csv: 500 rows, 271 columns
  batch_002.csv: 500 rows, 271 columns
  batch_003.csv: 40 rows, 271 columns

First batch columns (271):
['the_geom', 'points_id', 'location_name', 'input_address', 'match_address', 'latitude', 'longitude', 'major_basin_name', 'minor_basin_name', 'aquifer_name'] ...


In [90]:
WRI_SCORE_COLUMNS = [
    "bws_score",
    "bwd_score",
    "iav_score",
    "sev_score",
    "gtd_score",
    "rfr_score",
    "cfr_score",
    "drr_score",
    "ucw_score",
    "cep_score",
    "udw_score",
    "usa_score",
    "rri_score",
]

WRI_COLUMNS = ["points_id", "location_name", "latitude", "longitude"] + WRI_SCORE_COLUMNS

batch_dfs = [pd.read_csv(path) for path in batch_paths]
wri_source_df = pd.concat(batch_dfs, ignore_index=True)

print("Row counts:")
print(f"  wri_aqueduct_df: {len(wri_aqueduct_df)}")
print(f"  wri_source_df (combined): {len(wri_source_df)}")
print(f"  Per batch: {[len(df) for df in batch_dfs]}")

row_count_match = len(wri_source_df) == len(wri_aqueduct_df)
print(f"\nRow count match: {row_count_match}")

expected_ids = set(wri_aqueduct_df["id"])
actual_ids = set(wri_source_df["points_id"])
missing_ids = expected_ids - actual_ids
extra_ids = actual_ids - expected_ids
duplicate_ids = wri_source_df.loc[wri_source_df["points_id"].duplicated(), "points_id"].unique()

print(f"\nID validation (wri_aqueduct_df['id'] vs wri_source_df['points_id']):")
print(f"  Missing IDs: {len(missing_ids)}")
print(f"  Extra IDs: {len(extra_ids)}")
print(f"  Duplicate points_id in combined df: {len(duplicate_ids)}")

if missing_ids:
    print(f"  Sample missing IDs: {sorted(missing_ids)[:10]}")
if extra_ids:
    print(f"  Sample extra IDs: {sorted(extra_ids)[:10]}")
if len(duplicate_ids):
    print(f"  Duplicate ID(s): {list(duplicate_ids)}")

order_match = wri_aqueduct_df["id"].tolist() == wri_source_df["points_id"].tolist()
print(f"\nInput order preserved in combined output: {order_match}")

wri_source_df = wri_source_df[WRI_COLUMNS].copy()

output_path = Path("data/wri_source.csv")
wri_source_df.to_csv(output_path, index=False)
print(f"\nSaved reduced dataframe to {output_path}")
print(f"  Rows: {len(wri_source_df)}")
print(f"  Columns ({len(WRI_COLUMNS)}): {WRI_COLUMNS}")

Row counts:
  wri_aqueduct_df: 1040
  wri_source_df (combined): 1040
  Per batch: [500, 500, 40]

Row count match: True

ID validation (wri_aqueduct_df['id'] vs wri_source_df['points_id']):
  Missing IDs: 0
  Extra IDs: 0
  Duplicate points_id in combined df: 1
  Duplicate ID(s): [np.int64(975064000)]

Input order preserved in combined output: False

Saved reduced dataframe to data/wri_source.csv
  Rows: 1040
  Columns (17): ['points_id', 'location_name', 'latitude', 'longitude', 'bws_score', 'bwd_score', 'iav_score', 'sev_score', 'gtd_score', 'rfr_score', 'cfr_score', 'drr_score', 'ucw_score', 'cep_score', 'udw_score', 'usa_score', 'rri_score']


In [91]:
WRI_SENTINEL = -9999.0

records = []
for col in WRI_SCORE_COLUMNS:
    s = wri_source_df[col]
    sentinel_count = (s == WRI_SENTINEL).sum()
    valid = s[s != WRI_SENTINEL]
    records.append(
        {
            "column": col,
            "sentinel_count": int(sentinel_count),
            "sentinel_pct": round(sentinel_count / len(s) * 100, 1),
            "n_valid": int(len(valid)),
            "min": round(float(valid.min()), 4),
            "max": round(float(valid.max()), 4),
            "mean": round(float(valid.mean()), 4),
            "std": round(float(valid.std()), 4),
        }
    )

wri_stats = pd.DataFrame(records).set_index("column")

# Flag columns with data quality issues
zero_var = wri_stats[wri_stats["std"] == 0].index.tolist()
high_sentinel = wri_stats[wri_stats["sentinel_pct"] > 20].index.tolist()

print("WRI Score Column Diagnostics")
print("=" * 60)
print(wri_stats.to_string())
print()

if high_sentinel:
    print(f"HIGH SENTINEL RATE (>20% missing): {high_sentinel}")
if zero_var:
    print(f"ZERO VARIANCE (constant, drop before modeling): {zero_var}")


WRI Score Column Diagnostics
           sentinel_count  sentinel_pct  n_valid     min     max    mean     std
column                                                                          
bws_score               0           0.0     1040  0.0000  5.0000  2.1269  1.7639
bwd_score               0           0.0     1040  0.0040  5.0000  1.3067  1.4189
iav_score               0           0.0     1040  0.4399  4.7401  2.1653  0.8450
sev_score               0           0.0     1040  0.1413  2.9154  1.0703  0.6219
gtd_score             780          75.0      260  1.0644  3.2979  1.7218  0.7587
rfr_score               0           0.0     1040  0.0001  3.4068  0.8741  0.8692
cfr_score               0           0.0     1040  0.0000  2.6971  0.1819  0.4960
drr_score               0           0.0     1040  1.0109  2.9249  1.9728  0.3714
ucw_score               0           0.0     1040  0.8807  0.8807  0.8807  0.0000
cep_score               0           0.0     1040  0.9463  4.2903  2.8729  0.9225

### Feature Pruning: Dropping Low-Quality WRI Indicators

Two columns are removed before modeling based on the diagnostics above:

- **`gtd_score`** (Groundwater Table Decline): 75% of records (780/1040) carry the WRI sentinel value `-9999`, meaning WRI Aqueduct does not publish groundwater depletion estimates for the majority of US locations in this dataset. With only 260 valid values, any imputation would introduce more noise than signal.
- **`ucw_score`** (Untreated Collected Wastewater): Constant at `0.8807` across all 1,040 records — zero variance. A constant feature contributes nothing to a model and will be assigned zero importance by any tree-based or linear method.

> **Note:** `rri_score` (Riverine Flood Risk Index) is also constant at `0.88` across all records and carries the same zero-variance issue. It is dropped here as well.


In [92]:
DROP_COLUMNS = ["gtd_score", "ucw_score", "rri_score"]
WRI_SCORE_COLUMNS = [col for col in WRI_SCORE_COLUMNS if col not in DROP_COLUMNS]

print(f"Dropped: {DROP_COLUMNS}")
print(f"Remaining WRI score columns ({len(WRI_SCORE_COLUMNS)}): {WRI_SCORE_COLUMNS}")


Dropped: ['gtd_score', 'ucw_score', 'rri_score']
Remaining WRI score columns (10): ['bws_score', 'bwd_score', 'iav_score', 'sev_score', 'rfr_score', 'cfr_score', 'drr_score', 'cep_score', 'udw_score', 'usa_score']


## Build ML Feature Set

In [93]:
wri_for_join = wri_source_df.rename(
    columns={
        "points_id": "id",
        "location_name": "name",
        "latitude": "lat",
        "longitude": "lon",
    }
)

buildings_join = df_buildings[["id", "name", "lon", "lat", "impact_tier", "impact_tier_label", "est_mw"]].copy()
wri_scores = wri_for_join[["id"] + WRI_SCORE_COLUMNS].copy()

buildings_join = buildings_join.drop_duplicates(subset=["id"], keep="first")
wri_scores = wri_scores.drop_duplicates(subset=["id"], keep="first")

df_ml = buildings_join.merge(
    wri_scores,
    on="id",
    how="inner",
    validate="one_to_one",
)

# Validate metadata alignment between building records and WRI output
metadata_check = buildings_join.merge(
    wri_for_join[["id", "name", "lon", "lat"]],
    on="id",
    suffixes=("_building", "_wri"),
    how="inner",
)
for col in ["lat", "lon"]:
    metadata_check[f"{col}_building"] = metadata_check[f"{col}_building"].round(6)
    metadata_check[f"{col}_wri"] = metadata_check[f"{col}_wri"].round(6)

metadata_check["name_building"] = (
    metadata_check["name_building"].fillna("").astype(str).str.strip()
)
metadata_check["name_wri"] = metadata_check["name_wri"].fillna("").astype(str).str.strip()

metadata_mismatches = metadata_check[
    (metadata_check["name_building"] != metadata_check["name_wri"])
    | (metadata_check["lat_building"] != metadata_check["lat_wri"])
    | (metadata_check["lon_building"] != metadata_check["lon_wri"])
]

TARGET_COLUMN = "impact_tier"
FEATURE_COLUMNS = WRI_SCORE_COLUMNS
df_ml = df_ml[["id", "name", "lon", "lat", "impact_tier", "impact_tier_label", "est_mw"] + FEATURE_COLUMNS]

print(f"Joined rows: {len(df_ml)} / {len(df_buildings)} buildings")
print(f"Metadata mismatches on shared IDs: {len(metadata_mismatches)}")
print(f"Feature set shape: {df_ml[FEATURE_COLUMNS].shape}")
print(f"Target: {TARGET_COLUMN}")
print(f"Feature columns ({len(FEATURE_COLUMNS)}):")
print(FEATURE_COLUMNS)
df_ml.head(25)

Joined rows: 1039 / 1040 buildings
Metadata mismatches on shared IDs: 7
Feature set shape: (1039, 11)
Target: impact_tier
Feature columns (11):
['est_mw', 'bws_score', 'bwd_score', 'iav_score', 'sev_score', 'rfr_score', 'cfr_score', 'drr_score', 'cep_score', 'udw_score', 'usa_score']


,id,name,lon,lat,impact_tier,impact_tier_label,est_mw,bws_score,bwd_score,iav_score,sev_score,rfr_score,cfr_score,drr_score,cep_score,udw_score,usa_score
0,2744301,Verizon,-74.496521,40.544256,1,Colocation,15.86790,3.874239,1.728125,1.593892,0.905566,0.161611,0.000000,1.720637,4.037691,0.186522,0.0
1,7805491,Discover Financial Services New Albany,-82.814358,40.100657,1,Colocation,28.23135,2.834326,1.170674,1.012947,0.623887,0.106682,0.000000,2.509289,2.934937,0.273142,0.0
2,14930068,HPC (880),-106.542822,35.049942,1,Colocation,23.76945,3.577024,1.448563,1.658749,1.472403,1.592781,0.000000,1.530101,2.007740,0.529826,0.0
3,15884451,Barclays Datacenter,-74.285481,40.643753,1,Colocation,14.15595,4.010867,1.566559,0.744697,0.144982,0.333986,0.177306,1.783266,3.701583,0.000000,0.0
4,16282459,CyberNAP Glen Burnie,-76.606431,39.140242,1,Colocation,13.48185,1.126948,0.512190,1.964271,1.302858,0.572969,0.383627,2.066806,4.089767,0.000000,0.0
5,25706343,Cyxtera Dallas-Fort Worth Data Center,-97.041042,32.832718,2,Hyperscale,46.36560,4.211042,1.751433,3.033621,0.780655,0.559927,0.000000,2.279140,2.834903,0.000000,0.0
6,28544618,Equinix Infomart,-96.819511,32.800943,2,Hyperscale,35.97525,1.849822,0.738973,2.488517,0.842374,0.706051,0.000000,2.255284,3.498271,0.000000,0.0
7,29827954,CoreSite BO1,-71.079758,42.376706,1,Colocation,20.09400,2.188357,1.034589,0.439879,0.187774,2.078638,1.537531,1.418895,4.012761,0.087890,0.0
8,30666790,USPO Terminal Annex,-118.235393,34.058102,1,Colocation,14.89050,5.000000,4.773056,2.857265,1.422306,1.008137,0.005011,1.835201,4.053975,0.000000,0.0
9,31250931,CyrusOne Cincinnati-Mason CIN3,-84.311580,39.303896,1,Colocation,12.42195,0.000000,0.143947,1.074486,0.654621,0.539748,0.000000,2.412360,2.934831,0.624317,0.0


In [94]:
DROP_META = ["id", "name", "impact_tier_label", "sqft", "est_mw", "lon", "lat"]
drop_present = [c for c in DROP_META if c in df_ml.columns]

df_model = df_ml.drop(columns=drop_present)

assert list(df_model.columns) == [TARGET_COLUMN] + FEATURE_COLUMNS, (
    f"Unexpected columns: {list(df_model.columns)}"
)

print(f"Model-ready shape:  {df_model.shape}  →  {len(FEATURE_COLUMNS)} water risk features + 1 target")
print(f"\nDropped metadata:   {drop_present}")
print(f"\nFeatures ({len(FEATURE_COLUMNS)}):  {FEATURE_COLUMNS}")
print(f"Target:             {TARGET_COLUMN}")
print(f"\nTarget distribution:")
print(df_model[TARGET_COLUMN].map(TIER_LABELS).value_counts().reindex([TIER_LABELS[k] for k in sorted(TIER_LABELS)]).to_string())
df_model.head()


Model-ready shape:  (1039, 12)  →  11 water risk features + 1 target

Dropped metadata:   ['id', 'name', 'impact_tier_label', 'lon', 'lat']

Features (11):  ['est_mw', 'bws_score', 'bwd_score', 'iav_score', 'sev_score', 'rfr_score', 'cfr_score', 'drr_score', 'cep_score', 'udw_score', 'usa_score']
Target:             impact_tier

Target distribution:
impact_tier
Edge/Enterprise    181
Colocation         592
Hyperscale         266


,impact_tier,est_mw,bws_score,bwd_score,iav_score,sev_score,rfr_score,cfr_score,drr_score,cep_score,udw_score,usa_score
0,1,15.86790,3.874239,1.728125,1.593892,0.905566,0.161611,0.000000,1.720637,4.037691,0.186522,0.0
1,1,28.23135,2.834326,1.170674,1.012947,0.623887,0.106682,0.000000,2.509289,2.934937,0.273142,0.0
2,1,23.76945,3.577024,1.448563,1.658749,1.472403,1.592781,0.000000,1.530101,2.007740,0.529826,0.0
3,1,14.15595,4.010867,1.566559,0.744697,0.144982,0.333986,0.177306,1.783266,3.701583,0.000000,0.0
4,1,13.48185,1.126948,0.512190,1.964271,1.302858,0.572969,0.383627,2.066806,4.089767,0.000000,0.0


## Model Training

`df_model` is now fully preprocessed: 10 WRI water risk features and a 3-class `impact_tier` target across 1,039 buildings. Before fitting any model, the data is split into an 80/20 train/test partition.

A **stratified split** is used to preserve the class distribution across both sets — this matters here because the tiers are imbalanced (Edge/Enterprise ~17%, Colocation ~57%, Hyperscale ~26%). A random split risks leaving one tier under-represented in the test set.


In [95]:
from sklearn.model_selection import train_test_split

X = df_model[FEATURE_COLUMNS]
y = df_model[TARGET_COLUMN]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print(f"Train: {X_train.shape[0]} samples  |  Test: {X_test.shape[0]} samples")
print()
print("Class distribution:")
for label, tier_name in sorted(TIER_LABELS.items()):
    n_train = (y_train == label).sum()
    n_test  = (y_test  == label).sum()
    print(f"  {tier_name:<20} train={n_train:>3}  ({n_train/len(y_train)*100:.1f}%)   test={n_test:>3}  ({n_test/len(y_test)*100:.1f}%)")


Train: 831 samples  |  Test: 208 samples

Class distribution:
  Edge/Enterprise      train=145  (17.4%)   test= 36  (17.3%)
  Colocation           train=473  (56.9%)   test=119  (57.2%)
  Hyperscale           train=213  (25.6%)   test= 53  (25.5%)


In [96]:
import numpy as np

assert X_train.shape == (831, 10), f"Unexpected train shape: {X_train.shape}"
assert X_test.shape  == (208, 10), f"Unexpected test shape:  {X_test.shape}"
assert set(y_train.unique()).issubset({0, 1, 2}), "Unexpected target values in y_train"
assert set(y_test.unique()).issubset({0, 1, 2}),  "Unexpected target values in y_test"

nan_train      = X_train.isna().sum().sum()
nan_test       = X_test.isna().sum().sum()
sentinel_train = (X_train == WRI_SENTINEL).sum().sum()
sentinel_test  = (X_test  == WRI_SENTINEL).sum().sum()
inf_train      = np.isinf(X_train.values).sum()
inf_test       = np.isinf(X_test.values).sum()

print(f"Train shape:          {X_train.shape}")
print(f"Test shape:           {X_test.shape}")
print(f"NaN          — train: {nan_train},  test: {nan_test}")
print(f"Sentinel (-9999) — train: {sentinel_train},  test: {sentinel_test}")
print(f"Inf          — train: {inf_train},  test: {inf_test}")

assert nan_train == 0 and nan_test == 0,           "NaNs present — impute before training"
assert sentinel_train == 0 and sentinel_test == 0, "Sentinel values remain — check feature pruning"
assert inf_train == 0 and inf_test == 0,           "Inf values present — check source data"

print("\nAll checks passed. Ready to train.")


Train shape:          (831, 11)
Test shape:           (208, 11)
NaN          — train: 0,  test: 0
Sentinel (-9999) — train: 0,  test: 0
Inf          — train: 0,  test: 0

All checks passed. Ready to train.


In [97]:
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
from sklearn.utils.class_weight import compute_sample_weight

# XGBoost has no "batch size" (it is not minibatch-based). The highest-leverage
# knobs are the capacity/shrinkage tradeoff (max_depth, learning_rate, n_estimators)
# and regularization (min_child_weight, subsample, colsample_bytree, reg_*).
# Class imbalance is handled with inverse-frequency sample weights, since XGBoost
# — unlike CatBoost — does not auto-balance. Weights are recomputed per CV fold.
xgb_configs = {
    "shallow_slow": dict(n_estimators=600, max_depth=3, learning_rate=0.03,
                         subsample=0.8, colsample_bytree=0.8, min_child_weight=3),
    "balanced":     dict(n_estimators=300, max_depth=5, learning_rate=0.05,
                         subsample=0.9, colsample_bytree=0.8, min_child_weight=1),
    "deep_fast":    dict(n_estimators=200, max_depth=8, learning_rate=0.10,
                         subsample=0.7, colsample_bytree=0.7, min_child_weight=1),
    "regularized":  dict(n_estimators=400, max_depth=4, learning_rate=0.05,
                         subsample=0.8, colsample_bytree=0.8, min_child_weight=5,
                         reg_alpha=0.5, reg_lambda=2.0),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
xgb_models, xgb_cv_scores = {}, {}

for name, params in xgb_configs.items():
    fold_f1 = []
    for tr_idx, va_idx in cv.split(X_train, y_train):
        X_tr, X_va = X_train.iloc[tr_idx], X_train.iloc[va_idx]
        y_tr, y_va = y_train.iloc[tr_idx], y_train.iloc[va_idx]
        w_tr = compute_sample_weight("balanced", y_tr)
        m = XGBClassifier(**params, eval_metric="mlogloss", random_state=42, verbosity=0)
        m.fit(X_tr, y_tr, sample_weight=w_tr)
        fold_f1.append(f1_score(y_va, m.predict(X_va), average="macro"))
    fold_f1 = np.array(fold_f1)
    xgb_cv_scores[name] = fold_f1.mean()

    # Refit on the full training set with balanced weights
    full_w = compute_sample_weight("balanced", y_train)
    best = XGBClassifier(**params, eval_metric="mlogloss", random_state=42, verbosity=0)
    best.fit(X_train, y_train, sample_weight=full_w)
    xgb_models[name] = best
    print(f"{name:14s} CV macro-F1: {fold_f1.mean():.3f} ± {fold_f1.std():.3f}")

best_xgb_name = max(xgb_cv_scores, key=xgb_cv_scores.get)
xgb_model = xgb_models[best_xgb_name]
print(f"\nBest XGBoost config: '{best_xgb_name}' "
      f"(CV macro-F1 {xgb_cv_scores[best_xgb_name]:.3f}) -> assigned to xgb_model")


shallow_slow   CV macro-F1: 0.994 ± 0.003
balanced       CV macro-F1: 0.996 ± 0.003
deep_fast      CV macro-F1: 0.996 ± 0.003
regularized    CV macro-F1: 0.994 ± 0.005

Best XGBoost config: 'balanced' (CV macro-F1 0.996) -> assigned to xgb_model


In [98]:
from catboost import CatBoostClassifier
from sklearn.model_selection import cross_val_score

# CatBoost's highest-leverage knobs are learning_rate x iterations, tree depth
# (oblivious/symmetric trees, typically 4-10), and l2_leaf_reg — its primary
# regularizer. Imbalance is handled internally via auto_class_weights="Balanced",
# so no manual sample weighting is needed and cross_val_score is leakage-safe.
cb_configs = {
    "shallow_slow": dict(iterations=600, depth=4, learning_rate=0.03, l2_leaf_reg=3.0),
    "balanced":     dict(iterations=300, depth=6, learning_rate=0.05, l2_leaf_reg=3.0),
    "deep_fast":    dict(iterations=200, depth=8, learning_rate=0.10, l2_leaf_reg=1.0),
    "regularized":  dict(iterations=400, depth=5, learning_rate=0.05, l2_leaf_reg=8.0),
}

cb_models, cb_cv_scores = {}, {}

for name, params in cb_configs.items():
    m = CatBoostClassifier(**params, auto_class_weights="Balanced",
                           random_seed=42, verbose=0)
    scores = cross_val_score(m, X_train, y_train, cv=cv, scoring="f1_macro")
    cb_cv_scores[name] = scores.mean()

    best = CatBoostClassifier(**params, auto_class_weights="Balanced",
                              random_seed=42, verbose=0)
    best.fit(X_train, y_train)
    cb_models[name] = best
    print(f"{name:14s} CV macro-F1: {scores.mean():.3f} ± {scores.std():.3f}")

best_cb_name = max(cb_cv_scores, key=cb_cv_scores.get)
cb_model = cb_models[best_cb_name]
print(f"\nBest CatBoost config: '{best_cb_name}' "
      f"(CV macro-F1 {cb_cv_scores[best_cb_name]:.3f}) -> assigned to cb_model")


shallow_slow   CV macro-F1: 0.993 ± 0.004
balanced       CV macro-F1: 0.994 ± 0.005
deep_fast      CV macro-F1: 0.997 ± 0.003
regularized    CV macro-F1: 0.994 ± 0.007

Best CatBoost config: 'deep_fast' (CV macro-F1 0.997) -> assigned to cb_model


In [99]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import plotly.figure_factory as ff

tier_names = [TIER_LABELS[k] for k in sorted(TIER_LABELS)]

for model_name, model in [("XGBoost", xgb_model), ("CatBoost", cb_model)]:
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)

    print(f"{'─' * 50}")
    print(f"  {model_name}   (test accuracy: {acc:.3f})")
    print(f"{'─' * 50}")
    print(classification_report(y_test, y_pred, target_names=tier_names))

    cm = confusion_matrix(y_test, y_pred)
    fig = ff.create_annotated_heatmap(
        cm,
        x=tier_names,
        y=tier_names,
        colorscale="Blues",
        showscale=True,
    )
    fig.update_layout(
        title=f"{model_name} — Confusion Matrix (Test Set)",
        xaxis_title="Predicted",
        yaxis_title="Actual",
        yaxis={"autorange": "reversed"},
        height=420,
        margin={"t": 60},
    )
    fig.show()


──────────────────────────────────────────────────
  XGBoost   (test accuracy: 0.995)
──────────────────────────────────────────────────
                 precision    recall  f1-score   support

Edge/Enterprise       0.97      1.00      0.99        36
     Colocation       1.00      0.99      1.00       119
     Hyperscale       1.00      1.00      1.00        53

       accuracy                           1.00       208
      macro avg       0.99      1.00      0.99       208
   weighted avg       1.00      1.00      1.00       208



──────────────────────────────────────────────────
  CatBoost   (test accuracy: 1.000)
──────────────────────────────────────────────────
                 precision    recall  f1-score   support

Edge/Enterprise       1.00      1.00      1.00        36
     Colocation       1.00      1.00      1.00       119
     Hyperscale       1.00      1.00      1.00        53

       accuracy                           1.00       208
      macro avg       1.00      1.00      1.00       208
   weighted avg       1.00      1.00      1.00       208



In [100]:
for model_name, importances in [
    ("XGBoost",  xgb_model.feature_importances_),
    ("CatBoost", cb_model.get_feature_importance()),
]:
    fi = pd.DataFrame({"feature": FEATURE_COLUMNS, "importance": importances})
    fi = fi.sort_values("importance", ascending=True)

    fig = px.bar(
        fi,
        x="importance",
        y="feature",
        orientation="h",
        title=f"{model_name} — Feature Importance",
        labels={"importance": "Importance Score", "feature": "WRI Indicator"},
    )
    fig.update_layout(plot_bgcolor="white", height=420, margin={"l": 120, "t": 60})
    fig.show()
